## Synthetic Data Generation

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

current_folder = globals()['_dh'][0]
if str(current_folder.parent) not in sys.path:
    sys.path.append(str(current_folder.parent))

sys.path

['C:\\Program Files\\Python\\Python3.14\\python314.zip',
 'C:\\Program Files\\Python\\Python3.14\\DLLs',
 'C:\\Program Files\\Python\\Python3.14\\Lib',
 'C:\\Program Files\\Python\\Python3.14',
 'd:\\Source\\svm-using-hmm\\venv',
 '',
 'd:\\Source\\svm-using-hmm\\venv\\Lib\\site-packages',
 'd:\\Source\\svm-using-hmm']

In [4]:
from datalink.simulator import generate_synthetic_log_returns
from utils.datastructs import SVMParameters
from scipy import stats
import numpy as np
from inference.estimator import Hyperparameters, FastBayesianSVMEstimator

params = SVMParameters(beta0=0.2, beta1=0.07, beta2=-0.18, mu=0.1, phi=0.98, sigma_eta=0.1)

data = generate_synthetic_log_returns(params, n=300)

# 1. Define the SMN log-density function (e.g., Student-t distribution)
def student_t_logpdf(y: float, mu: np.ndarray, sigma: np.ndarray, nu: float) -> np.ndarray:
    """
    Log-density of the Student-t distribution parameterized by location, scale, and df.
    """
    # Using scipy stats, passing arrays for vectorized operations over the grid
    return stats.t.logpdf(y, df=nu, loc=mu, scale=sigma)

# 2. Generate/Load Mock Data (1000 days of random returns for demonstration)
np.random.seed(42)
mock_returns = data['Log_Return'].to_numpy()

# 3. Configure Hyperparameters
config = Hyperparameters(
    m=50,               # Lowered to 50 for speed in MWE, paper uses 100-200
    b_limit=2.5,        # Grid boundaries
    is_samples=500      # Samples for IS
)

# 4. Instantiate and run the estimator
estimator = FastBayesianSVMEstimator(
    data=mock_returns,
    smn_logpdf=student_t_logpdf,
    hyperparams=config
)

# 5. Extract Results
result = estimator.estimate()

print("\n--- ESTIMATION RESULTS ---")
print(f"Optimization Success: {result.success}")
print("\nPosterior Means (Constrained Space):")
for param, value in result.posterior_mean_con.items():
    print(f"{param:>10}: {value:.4f}")

2026-08-27 13:36:04,828 [INFO] SVMEstimator: Starting numerical maximization (L-BFGS-B)...
d:\Source\svm-using-hmm\inference\estimator.py:188: OptimizeWarning: Unknown solver options: disp
  opt_res = minimize(
2026-08-27 13:36:18,528 [INFO] SVMEstimator: MAP optimization complete. Unconstrained Mode: [ 0.53   0.302 -0.508  0.42   4.63  -3.297 -0.388]
2026-08-27 13:36:18,530 [INFO] SVMEstimator: Drawing 500 samples for Importance Sampling inference...
2026-08-27 13:36:39,176 [INFO] SVMEstimator: Importance Sampling complete.



--- ESTIMATION RESULTS ---
Optimization Success: True

Posterior Means (Constrained Space):
     beta0: 0.6486
     beta1: 0.1903
     beta2: -0.6072
        mu: 0.5656
       phi: 0.9852
 sigma_eta: 0.0408
        nu: 28.6533
